# 0.2 — Torch projected-gradient baseline

This is the minimal numerical feasibility test. It validates a Torch-only non-negative projected-gradient solver on known sparse synthetic mixtures created from a seeded subset of the global molecular catalogue.

## Method

`deconvolution.projected_gradient_baseline` first selects exactly 100 candidate ions from the configured global condition, then renders only that local `K`. It reports per-spectrum convergence and an autograd finite-difference check. No CPU SciPy solver is used.

In [ ]:
from pathlib import Path

import pandas as pd
from msi_autoencoder_wrapper.analysis.precompute.cli import run_precompute_command
from msi_autoencoder_wrapper.visualization.metrics import plot_metric_distribution

REPOSITORY_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'pyproject.toml').is_file())
NOTEBOOK_DIR = REPOSITORY_ROOT / 'assets/experiments/autoencoder_architecture/notebooks/annotation_model/19_09_deconvolution_inital'
SETTINGS_PATH = NOTEBOOK_DIR / 'analysis_settings.yaml'
RESULTS_DIR = NOTEBOOK_DIR / 'part_0_2_projected_gradient_baseline_results'
print(run_precompute_command(SETTINGS_PATH, background=False))

## Reproducible execution

Run the printed precompute command before evaluating these cells. The persisted `selected_candidates.csv` fixes the 100 catalogue columns used by every reported number.

In [ ]:
convergence_path = RESULTS_DIR / 'convergence.csv'
gradient_path = RESULTS_DIR / 'gradient_validation.csv'
if not convergence_path.is_file() or not gradient_path.is_file():
    raise FileNotFoundError('Missing baseline artifacts. Run the command printed above.')
convergence = pd.read_csv(convergence_path)
gradient_validation = pd.read_csv(gradient_path)
display(gradient_validation)
convergence.groupby('iterations', as_index=False).agg(
    objective=('objective', 'mean'),
    abundance_mae=('abundance_mae', 'mean'),
    reconstruction_mse=('reconstruction_mse', 'mean'),
    support_exact_fraction=('support_exact', 'mean'),
)

In [ ]:
final_iterations = int(convergence['iterations'].max())
final_objectives = convergence.loc[convergence['iterations'].eq(final_iterations), 'objective'].to_numpy()
figure, axis = plot_metric_distribution(
    final_objectives,
    metric=f'Objective after {final_iterations} projected-gradient updates',
    bins=20,
    label='synthetic spectra',
)
figure.savefig(RESULTS_DIR / 'objective_distribution.png', bbox_inches='tight')
figure

## Decision criterion

`gradcheck_passed` must be true and the mean objective must not increase with the configured update count. Only then does it make sense to interpret recovery failures in the identifiability experiment rather than treating them as a numerical implementation error.